# Convolutional Neural Networks (CNNs)

**CNN**: it's a specialized class of deep neural networks designed primarily for processing structured grid-like data, such as images, videos, and time-series signals. They excel at capturing spatial hierarchies and local patterns through weight-sharing and translation invariance, making them highly efficient for tasks involving visual data. Unlike fully connected networks, CNNs use fewer parameters by exploiting the local connectivity of data (e.g., nearby pixels in an image are more correlated than distant ones).

## Core Components of CNNs

### Convolutional Layers

- Primary building block: Applies learnable filters (kernels) to input to extract features.
- Operation: Sliding kernel over input, computing dot products (cross-correlation technically).
- Key hyperparameters:
  - Kernel size (e.g., 3×3, 5×5) - Smaller kernels preferred in modern designs for depth.
  - Number of filters - Determines output channels (feature maps).
  - Stride - Step size of kernel slide (default 1; higher reduces spatial dimensions).
  - Padding - Adds zeros around input ("valid" = no padding; "same" = preserves spatial size).
- Output feature map size formula (for one dimension):
  $ O = \left\lfloor \frac{I - K + 2P}{S} \right\rfloor + 1 $
  where $I$ = input size, $K$ = kernel size, $P$ = padding, $S$ = stride.
- Multi-channel inputs (e.g., RGB): Kernel has depth matching input channels; output channels = number of kernels.
- Parameter sharing: Same kernel applied across entire input → translation invariance and drastic parameter reduction.

### Activation Functions

- Applied element-wise after convolution to introduce non-linearity.
- Common choices:
  - ReLU (Rectified Linear Unit): $f(x) = \max(0, x)$ - Fast, mitigates vanishing gradients; variants: Leaky ReLU, Parametric ReLU, ELU.
  - Sigmoid/Tanh - Rarely used in hidden layers due to saturation/vanishing gradients.
  - Swish/GELU - Used in modern architectures for smoother gradients.
- Placement: Typically after each convolution (Conv → Activation → Pooling pattern).

### Pooling Layers

- Downsampling operation to reduce spatial dimensions, introduce translation invariance, and control overfitting.
- Types:
  - Max Pooling: Takes maximum value in window - Preserves strong features.
  - Average Pooling: Takes mean - Smoother downsampling.
  - Global Average Pooling (GAP): Pools entire feature map to single value per channel - Common in modern nets to replace FC layers.
- Hyperparameters: Pool size (e.g., 2×2), stride (often = pool size for non-overlapping).
- Benefits: Reduces computation, increases receptive field.

### Fully Connected (Dense) Layers

- Placed at the end (classifier head).
- Flattens feature maps and applies standard matrix multiplication.
- Modern trend: Replace with Global Average Pooling + single dense layer to reduce parameters (e.g., in ResNet, EfficientNet).
- Often followed by softmax/sigmoid for classification probabilities.

### Auxiliary Components

- **Batch Normalization (BN)**: Normalizes activations per mini-batch → Stabilizes training, allows higher learning rates.
- **Dropout**: Randomly zeros neurons during training → Regularization.
- **Skip/Residual Connections**: Add input to layer output (ResNet) → Enables very deep training.
- **Inception Modules**: Parallel convolutions of different sizes + concatenation.
- **Depthwise Separable Convolutions**: Used in MobileNet - Depthwise conv + pointwise conv for efficiency.

## How CNNs Work

```mermaid
graph LR
image(Input: Image)

subgraph backbone[Backbone<br/>Extracts features from multiple resolutions<br/>Conv layers, Pooling]
    direction TB
    backbone1(backbone) e1@--> backbone2(backbone)
    backbone2(backbone) e2@--> backbone3(backbone)
end

image e3@--> backbone

subgraph neck[Neck<br/>Performs multi-resolution feature aggregation<br/>FPN]
    direction TB
    neck1(neck) e4@--> neck2(neck)
    neck2(neck) e5@--> neck1
    neck2(neck) e6@--> neck3(neck)
    neck3(neck) e7@--> neck2
end

backbone1 e8@--> neck1
backbone2 e9@--> neck2
backbone3 e10@--> neck3

subgraph head[Head<br/>Generates final predictions based on object resolution<br/>Classification, Detection]
    direction TB
    head1(head)
    head2(head)
    head3(head)
end

neck1 e11@--> head1
neck2 e12@--> head2
neck3 e13@--> head3

e1@{ animate: true }
e2@{ animate: true }
e3@{ animate: true }
e4@{ animate: true }
e5@{ animate: true }
e6@{ animate: true }
e7@{ animate: true }
e8@{ animate: true }
e9@{ animate: true }
e10@{ animate: true }
e11@{ animate: true }
e12@{ animate: true }
e13@{ animate: true }
```

- **Early Layers**: Detect low-level features (edges, corners, textures) via small receptive fields.
- **Middle Layers**: Combine into mid-level patterns (shapes, parts of objects).
- **Deeper Layers**: High-level semantic features (object parts, full objects).
- **Receptive Field Growth**: Each layer expands effective receptive field through stacking conv/pooling.
- **Translation Invariance**: Pooling and shared weights make predictions robust to small shifts.
- **Parameter Efficiency**: For a 224×224×3 input, a dense net would have billions of parameters; CNNs have millions or fewer.
